In [3]:
import pandas as pd

# Load CSV
df = pd.read_csv('/content/02_nav_history.csv')

# 1. Parse date column to datetime
df['date'] = pd.to_datetime(df['date'], errors='coerce')

# 2. Remove duplicate rows
df = df.drop_duplicates()

# 3. Sort by amfi_code and date
df = df.sort_values(['amfi_code', 'date'])

# 4. Forward-fill missing NAV values within each scheme
df['nav'] = df.groupby('amfi_code')['nav'].ffill()

# 5. Validate NAV > 0
invalid_nav = df[df['nav'] <= 0]

print(f"Invalid NAV records: {len(invalid_nav)}")

# Optional: remove invalid NAV rows
df = df[df['nav'] > 0]

# 6. Check missing values after cleaning
print(df.isnull().sum())

# Save cleaned file
df.to_csv('cleaned_nav_history.csv', index=False)

print("Cleaning completed successfully!")

Invalid NAV records: 0
amfi_code    0
date         0
nav          0
dtype: int64
Cleaning completed successfully!


In [12]:
import pandas as pd

# Load CSV
transactions_df = pd.read_csv('/content/08_investor_transactions.csv')


In [13]:
# 1. Standardise transaction_type values (SIP/Lumpsum/Redemption)
# Convert to lowercase and strip whitespace to ensure consistency
transactions_df['transaction_type'] = transactions_df['transaction_type'].str.lower().str.strip()

# Define a mapping for standardization
transaction_type_mapping = {
    'sip': 'SIP',
    'lumpsum': 'Lumpsum',
    'redemption': 'Redemption'
}

# Apply the mapping
transactions_df['transaction_type'] = transactions_df['transaction_type'].replace(transaction_type_mapping)

# Report unmapped types if any
unmapped_types = transactions_df[~transactions_df['transaction_type'].isin(['SIP', 'Lumpsum', 'Redemption'])]['transaction_type'].unique()
if len(unmapped_types) > 0:
    print(f"Warning: Unmapped transaction types found: {unmapped_types}")


In [11]:
# 2. Validate amount > 0
# Convert 'amount_inr' to numeric, coercing errors to NaN
transactions_df['amount_inr'] = pd.to_numeric(transactions_df['amount_inr'], errors='coerce')

# Identify and report invalid/non-positive amounts
invalid_amounts_count = transactions_df['amount_inr'].isnull().sum() + (transactions_df['amount_inr'] <= 0).sum()
print(f"Number of invalid or non-positive amounts: {invalid_amounts_count}")

# Remove rows with invalid amounts
transactions_df = transactions_df[transactions_df['amount_inr'] > 0].copy()


Number of invalid or non-positive amounts: 0


In [14]:
# 3. Fix date formats
# Assuming 'transaction_date' is the date column, adjust if necessary
date_cols = ['transaction_date']

for col in date_cols:
    if col in transactions_df.columns:
        transactions_df[col] = pd.to_datetime(transactions_df[col], errors='coerce')
        missing_dates_count = transactions_df[col].isnull().sum()
        if missing_dates_count > 0:
            print(f"Warning: {missing_dates_count} invalid dates found in '{col}' after parsing. These rows now have NaT.")


In [15]:
# 4. Check KYC status enum values
# Convert to lowercase and strip whitespace for standardization
transactions_df['kyc_status'] = transactions_df['kyc_status'].astype(str).str.lower().str.strip()

# Define allowed KYC status values
allowed_kyc_statuses = ['completed', 'pending', 'rejected', 'in-progress'] # Add/adjust based on your expected values

# Identify and report unexpected KYC status values
unexpected_kyc_statuses = transactions_df[~transactions_df['kyc_status'].isin(allowed_kyc_statuses)]['kyc_status'].unique()
if len(unexpected_kyc_statuses) > 0:
    print(f"Warning: Unexpected KYC status values found: {unexpected_kyc_statuses}. Consider standardizing or correcting these.")


In [16]:
# Final check for missing values after all cleaning steps
print("\nMissing values after cleaning:")
print(transactions_df.isnull().sum())

# Display cleaned data info and head for verification
print("\nCleaned DataFrame Info:")
transactions_df.info()
print("\nCleaned DataFrame Head:")
display(transactions_df.head())

# Save the cleaned file
transactions_df.to_csv('cleaned_investor_transactions.csv', index=False)
print("\nCleaning of investor_transactions.csv completed successfully and saved to 'cleaned_investor_transactions.csv'.")



Missing values after cleaning:
investor_id           0
transaction_date      0
amfi_code             0
transaction_type      0
amount_inr            0
state                 0
city                  0
city_tier             0
age_group             0
gender                0
annual_income_lakh    0
payment_mode          0
kyc_status            0
dtype: int64

Cleaned DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32778 entries, 0 to 32777
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   investor_id         32778 non-null  object        
 1   transaction_date    32778 non-null  datetime64[ns]
 2   amfi_code           32778 non-null  int64         
 3   transaction_type    32778 non-null  object        
 4   amount_inr          32778 non-null  int64         
 5   state               32778 non-null  object        
 6   city                32778 non-null  object        
 7   

,investor_id,transaction_date,amfi_code,transaction_type,amount_inr,state,city,city_tier,age_group,gender,annual_income_lakh,payment_mode,kyc_status
0,INV003054,2024-01-01,119092,SIP,1834,Telangana,Hyderabad,T30,56+,Female,77.1,UPI,verified
1,INV002952,2024-01-01,148567,Redemption,392882,Punjab,Amritsar,B30,18-25,Male,7.1,Cheque,verified
2,INV003420,2024-01-01,118636,SIP,912,Haryana,Faridabad,B30,36-45,Male,47.2,Mandate,verified
3,INV003436,2024-01-01,118634,SIP,1102,Maharashtra,Mumbai,T30,36-45,Female,54.4,Cheque,pending
4,INV004691,2024-01-01,119094,Lumpsum,8682,Delhi,Noida,T30,26-35,Male,14.5,Net Banking,pending



Cleaning of investor_transactions.csv completed successfully and saved to 'cleaned_investor_transactions.csv'.


In [22]:
import pandas as pd

# Load CSV
scheme_df = pd.read_csv('/content/07_scheme_performance.csv')

print("Original Scheme Performance DataFrame Info:")
scheme_df.info()
print("\nOriginal Head:")
display(scheme_df.head())


Original Scheme Performance DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40 entries, 0 to 39
Data columns (total 19 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   amfi_code           40 non-null     int64  
 1   scheme_name         40 non-null     object 
 2   fund_house          40 non-null     object 
 3   category            40 non-null     object 
 4   plan                40 non-null     object 
 5   return_1yr_pct      40 non-null     float64
 6   return_3yr_pct      40 non-null     float64
 7   return_5yr_pct      40 non-null     float64
 8   benchmark_3yr_pct   40 non-null     float64
 9   alpha               40 non-null     float64
 10  beta                40 non-null     float64
 11  sharpe_ratio        40 non-null     float64
 12  sortino_ratio       40 non-null     float64
 13  std_dev_ann_pct     40 non-null     float64
 14  max_drawdown_pct    40 non-null     float64
 15  aum_crore      

,amfi_code,scheme_name,fund_house,category,plan,return_1yr_pct,return_3yr_pct,return_5yr_pct,benchmark_3yr_pct,alpha,beta,sharpe_ratio,sortino_ratio,std_dev_ann_pct,max_drawdown_pct,aum_crore,expense_ratio_pct,morningstar_rating,risk_grade
0,119551,SBI Bluechip Fund - Regular Plan - Growth,SBI Mutual Fund,Large Cap,Regular,12.42,12.36,14.45,11.49,0.87,0.89,0.88,1.29,14.0,-21.70,14288,1.54,4,Moderate
1,119552,SBI Bluechip Fund - Direct Plan - Growth,SBI Mutual Fund,Large Cap,Direct,15.25,11.30,14.23,9.52,1.78,0.87,0.81,1.29,14.0,-24.43,1231,0.66,3,Moderate
2,119598,SBI Small Cap Fund - Regular Plan - Growth,SBI Mutual Fund,Small Cap,Regular,24.56,23.39,20.67,22.16,1.23,0.89,0.94,1.35,25.0,-13.35,19259,1.43,5,Very High
3,119599,SBI Small Cap Fund - Direct Plan - Growth,SBI Mutual Fund,Small Cap,Direct,20.59,23.14,21.82,22.01,1.13,1.04,0.93,1.67,25.0,-24.78,36061,0.72,4,Very High
4,119120,SBI Magnum Gilt Fund - Regular Plan - Growth,SBI Mutual Fund,Gilt,Regular,5.34,6.07,5.43,4.47,1.60,0.22,1.52,2.11,4.0,-2.30,24101,0.77,5,Low


In [23]:
# 1. Validate all return values are numeric
# Identify columns that represent returns (e.g., '1Y Return', '3Y Return', '5Y Return', etc.)
# These column names might vary. Let's assume common patterns.

return_cols = [col for col in scheme_df.columns if 'return' in col.lower()]

for col in return_cols:
    if col in scheme_df.columns:
        # Convert to numeric, coercing errors to NaN
        scheme_df[col] = pd.to_numeric(scheme_df[col], errors='coerce')
        missing_returns = scheme_df[scheme_df[col].isnull()]
        if len(missing_returns) > 0:
            print(f"Warning: {len(missing_returns)} non-numeric or missing return values found in '{col}'. These have been converted to NaN.")

print("Return columns info after numeric conversion:")
for col in return_cols:
    if col in scheme_df.columns:
        print(f"Column '{col}': {scheme_df[col].dtype}")
display(scheme_df[return_cols].describe())


Return columns info after numeric conversion:
Column 'return_1yr_pct': float64
Column 'return_3yr_pct': float64
Column 'return_5yr_pct': float64


,return_1yr_pct,return_3yr_pct,return_5yr_pct
count,40.000000,40.000000,40.000000
mean,14.376000,14.089000,14.516750
std,4.883023,4.617253,4.454021
min,4.260000,5.140000,5.430000
25%,11.735000,12.035000,12.340000
50%,14.620000,14.205000,14.185000
75%,16.392500,15.882500,17.585000
max,24.930000,23.390000,23.800000


In [24]:
# 2. Flag anomalies in return values
# Anomalies can be identified as values significantly outside a certain range (e.g., beyond 3 standard deviations, or very high/low returns).
# We'll use a simple approach: flag returns that are extremely high or low.

anomaly_threshold_high = 1000  # e.g., returns > 1000% might be data entry errors
anomaly_threshold_low = -100   # e.g., returns < -100% might be data entry errors (a scheme cannot lose more than 100%)

scheme_df['is_return_anomaly'] = False

for col in return_cols:
    if col in scheme_df.columns and scheme_df[col].dtype in ['float64', 'int64']:
        anomalous_high = scheme_df[col] > anomaly_threshold_high
        anomalous_low = scheme_df[col] < anomaly_threshold_low
        scheme_df['is_return_anomaly'] = scheme_df['is_return_anomaly'] | anomalous_high | anomalous_low

anomaly_count = scheme_df['is_return_anomaly'].sum()
if anomaly_count > 0:
    print(f"Found {anomaly_count} schemes with potential return anomalies. Review these entries:")
    display(scheme_df[scheme_df['is_return_anomaly']][['scheme_name'] + return_cols])
else:
    print("No significant return anomalies detected based on defined thresholds.")


No significant return anomalies detected based on defined thresholds.


In [25]:
# 3. Check expense_ratio range (0.1% – 2.5%)
# Convert 'expense_ratio_pct' to numeric first
scheme_df['expense_ratio_pct'] = pd.to_numeric(scheme_df['expense_ratio_pct'], errors='coerce')

# Define the valid range (as decimals)
min_expense_ratio = 0.001  # 0.1%
max_expense_ratio = 0.025  # 2.5%

# Flag expense ratios outside the valid range
invalid_expense_ratio = scheme_df[
    scheme_df['expense_ratio_pct'].isnull() |
    (scheme_df['expense_ratio_pct'] < min_expense_ratio) |
    (scheme_df['expense_ratio_pct'] > max_expense_ratio)
]

if len(invalid_expense_ratio) > 0:
    print(f"Found {len(invalid_expense_ratio)} schemes with expense_ratio_pct outside the 0.1% - 2.5% range or non-numeric:")
    display(invalid_expense_ratio[['scheme_name', 'expense_ratio_pct']])
    # Optional: remove these rows, or impute, or cap values
    # For cleaning, we might want to cap them or set to NaN for further handling
    # For now, we just report.
else:
    print("All expense_ratio_pct values are within the 0.1% - 2.5% range and are numeric.")

print("Expense Ratio statistics:")
display(scheme_df['expense_ratio_pct'].describe())


KeyError: 'expense_ratio'

In [26]:
# Final check for missing values after cleaning
print("\nMissing values after cleaning (Scheme Performance):")
print(scheme_df.isnull().sum())

# Display cleaned data info and head
print("\nCleaned Scheme Performance DataFrame Info:")
scheme_df.info()
print("\nCleaned Scheme Performance DataFrame Head:")
display(scheme_df.head())

# Save the cleaned file
scheme_df.to_csv('cleaned_scheme_performance.csv', index=False)
print("\nCleaning of scheme_performance.csv completed successfully and saved to 'cleaned_scheme_performance.csv'.")



Missing values after cleaning (Scheme Performance):
amfi_code             0
scheme_name           0
fund_house            0
category              0
plan                  0
return_1yr_pct        0
return_3yr_pct        0
return_5yr_pct        0
benchmark_3yr_pct     0
alpha                 0
beta                  0
sharpe_ratio          0
sortino_ratio         0
std_dev_ann_pct       0
max_drawdown_pct      0
aum_crore             0
expense_ratio_pct     0
morningstar_rating    0
risk_grade            0
is_return_anomaly     0
dtype: int64

Cleaned Scheme Performance DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40 entries, 0 to 39
Data columns (total 20 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   amfi_code           40 non-null     int64  
 1   scheme_name         40 non-null     object 
 2   fund_house          40 non-null     object 
 3   category            40 non-null     object 
 4   plan    

,amfi_code,scheme_name,fund_house,category,plan,return_1yr_pct,return_3yr_pct,return_5yr_pct,benchmark_3yr_pct,alpha,beta,sharpe_ratio,sortino_ratio,std_dev_ann_pct,max_drawdown_pct,aum_crore,expense_ratio_pct,morningstar_rating,risk_grade,is_return_anomaly
0,119551,SBI Bluechip Fund - Regular Plan - Growth,SBI Mutual Fund,Large Cap,Regular,12.42,12.36,14.45,11.49,0.87,0.89,0.88,1.29,14.0,-21.70,14288,1.54,4,Moderate,False
1,119552,SBI Bluechip Fund - Direct Plan - Growth,SBI Mutual Fund,Large Cap,Direct,15.25,11.30,14.23,9.52,1.78,0.87,0.81,1.29,14.0,-24.43,1231,0.66,3,Moderate,False
2,119598,SBI Small Cap Fund - Regular Plan - Growth,SBI Mutual Fund,Small Cap,Regular,24.56,23.39,20.67,22.16,1.23,0.89,0.94,1.35,25.0,-13.35,19259,1.43,5,Very High,False
3,119599,SBI Small Cap Fund - Direct Plan - Growth,SBI Mutual Fund,Small Cap,Direct,20.59,23.14,21.82,22.01,1.13,1.04,0.93,1.67,25.0,-24.78,36061,0.72,4,Very High,False
4,119120,SBI Magnum Gilt Fund - Regular Plan - Growth,SBI Mutual Fund,Gilt,Regular,5.34,6.07,5.43,4.47,1.60,0.22,1.52,2.11,4.0,-2.30,24101,0.77,5,Low,False



Cleaning of scheme_performance.csv completed successfully and saved to 'cleaned_scheme_performance.csv'.
